# DRCFNet Model - CMU-MOSEI Dataset

Dynamic Reconstructive Context Fusion Network for Sentiment Analysis

In [ ]:
# !git clone https://github.com/M-Jafarkhani/Multimodal-Emotion-Recognition

In [ ]:
import gdown

file_id = "1zFOBHijVppTiyteSsi0aTFYPEsda_AOk"
destination = "mosei_raw.pkl"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}", destination, quiet=False)

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.append('../../')

from loader import get_dataloader
from models.drcfnet import DRCFNet
from objectives import NeuroSymbolicLoss
from training.train import train, test
from evaluation.performance import eval_affect

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

In [ ]:
# Load MOSEI Data
train_data, valid_data, test_data = get_dataloader(
    '/content/mosei_raw.pkl', 
    max_pad=True, 
    max_seq_len=50
)

In [ ]:
# Initialize Model and Loss
model = DRCFNet(dim_v=713, dim_a=74, dim_t=300, d=128, n_heads=4, dropout=0.2).to(device)
criterion = NeuroSymbolicLoss(lambda_logic=0.1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

In [ ]:
# Initialize Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)

# Train the model
model, history = train(
    model=model,
    train_loader=train_data,
    valid_loader=valid_data,
    criterion=criterion,
    optimizer=optimizer,
    epochs=50,
    device=device,
    scheduler=scheduler,
    clip_grad=1.0
)

In [ ]:
# Test the model
metrics, preds, labels = test(
    model=model,
    dataloader=test_data,
    criterion=criterion,
    device=device,
    return_preds=True
)

# Evaluate metrics
prede = []
oute = preds.cpu().numpy().tolist()
for i in oute:
    if i[0] > 0:
        prede.append(1)
    elif i[0] < 0:
        prede.append(-1)
    else:
        prede.append(0)
pred_classes = torch.LongTensor(prede)

accs = eval_affect(labels, pred_classes)
acc2 = eval_affect(labels, pred_classes, exclude_zero=False)

print(f"Recall: {accs*100:.4f}% | Total Accuracy: {acc2*100:.4f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["valid_loss"], label="Valid Loss")
plt.title("Total Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_task"], label="Train MAE")
plt.plot(history["valid_task"], label="Valid MAE")
plt.title("Task MAE (Regression)")
plt.legend()
plt.show()

In [ ]:
# Save the model
torch.save(model.state_dict(), "drcfnet_mosei_best.pt")
print("Model saved as drcfnet_mosei_best.pt")